# XGBoost Model

In [1]:
# Import libraries
import warnings;

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, make_scorer, confusion_matrix

import xgboost as xgb

warnings.filterwarnings("ignore")
sns.set()
sns.set_theme(style="whitegrid")

## Dataset

In [7]:
# Import dataset: dataset_logreg
df = pd.read_csv('dataset_tree.csv')
df.head()

,age,duration_latest,count_call_current,days_last_campaign,count_call_previous,evr_quarterly,cpi_monthly,cci_monthly,type_employment,highest_educ,month_last_contacted,previous_campaign,Response,civil_status_Cat_1_m***d,civil_status_Cat_2_s***e,civil_status_Cat_3_u***n,home_loan_Cat_1_u***n,home_loan_Cat_2_y***s,personal_loan_Cat_1_u***n,personal_loan_Cat_2_y***s,contact_medium_Cat_1_t***e,dow_last_contacted_Cat_1_m***n,dow_last_contacted_Cat_2_t***u,dow_last_contacted_Cat_3_t***e,dow_last_contacted_Cat_4_w***d,credit_facility_Cat_1_u***n
0,57,8.770292,0.000000,0,1,-1.8,92.893,-46.2,0.163871,0.229763,0.335429,1,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,55,8.227745,0.554826,0,0,1.1,93.994,-36.4,0.008372,0.042204,0.335429,2,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,33,5.142826,0.000000,0,1,-1.8,92.893,-46.2,0.224534,0.147898,0.335429,1,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,36,8.678318,0.904026,0,0,1.4,94.465,-41.8,0.253336,0.229763,0.129697,2,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,27,7.418466,0.554826,0,0,1.4,93.918,-42.7,0.025745,0.229763,0.175015,2,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Define features and target variable
X = df.drop('Response', axis=1)
y = df[['Response']]

# Ensure the target variable is binary
y = y.astype('int')

# Check the shape of the features and target variable
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Features shape: (34997, 25)
Target shape: (34997, 1)


## Modeling

In [ ]:
# Hyperparameters
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.1, 1, 10]
}

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss', 
    random_state=42
    )

# Create a GridSearchCV object
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',  # Use F1 score for evaluation
    n_jobs=-1
)
# Fit the model
grid.fit(X_train, y_train.values.ravel())

# Get the best parameters and score
best_params = grid.best_params_
best_score = grid.best_score_
print(f"Best parameters: {best_params}")
print(f"Best F1 score: {best_score:.4f}")

# Best estimator
best_model = grid.best_estimator_

Best parameters: {'colsample_bytree': 1.0, 'gamma': 0.1, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'reg_alpha': 0.1, 'reg_lambda': 10, 'subsample': 0.8}
Best F1 score: 0.5953


In [ ]:
random = RandomizedSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',  # Use F1 score for evaluation
    n_jobs=-1
)

## Evaluation

In [ ]:
# Making predictions
y_pred = best_model.predict(X_test)

# Evaluate the model
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])